In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/jonathantran/Desktop/nba_economics")
STAGING_DIR = PROJECT_ROOT / "data" / "staging"
CANONICAL_DIR = PROJECT_ROOT / "data" / "canonical"

s15 = pd.read_csv(
    STAGING_DIR / "front_office" / "S15_GM_Tenures.csv"
)

s16 = pd.read_csv(
    STAGING_DIR / "ownership" / "S16_Ownership_Tenures.csv"
)

print("S15:", s15.shape)
print("S16:", s16.shape)

S15: (181, 5)
S16: (70, 7)


In [2]:
for name, df in [("S15", s15), ("S16", s16)]:

    print(f"\n{'=' * 55}")
    print(name)
    print(f"{'=' * 55}")

    print("Columns:", df.columns.tolist())

    print("\nData types:")
    print(df.dtypes.to_string())

    print("\nMissing values:")
    print(df.isna().sum().to_string())

    print("\nSample records:")
    display(df.head(8))


S15
Columns: ['executive_name', 'start_date', 'end_date', 'franchise_id', 'franchise_name']

Data types:
executive_name    str
start_date        str
end_date          str
franchise_id      str
franchise_name    str

Missing values:
executive_name     0
start_date         0
end_date          30
franchise_id       0
franchise_name     0

Sample records:


,executive_name,start_date,end_date,franchise_id,franchise_name
0,Pete Babcock,1990-02-14,2003-04-02,ATL,Atlanta Hawks
1,Billy Knight,2003-04-02,2008-05-28,ATL,Atlanta Hawks
2,Rick Sund,2008-05-28,2012-06-25,ATL,Atlanta Hawks
3,Danny Ferry,2012-06-25,2014-09-12,ATL,Atlanta Hawks
4,Mike Budenholzer,2014-09-12,2017-05-25,ATL,Atlanta Hawks
5,Travis Schlenk,2017-05-25,2022-12-21,ATL,Atlanta Hawks
6,Landry Fields,2022-12-21,2025-04-21,ATL,Atlanta Hawks
7,Onsi Saleh,2025-04-21,NaN,ATL,Atlanta Hawks



S16
Columns: ['owner_name', 'team_id', 'start_season', 'end_season', 'start_year', 'end_year', 'source']

Data types:
owner_name          str
team_id             str
start_season        str
end_season          str
start_year        int64
end_year        float64
source              str

Missing values:
owner_name       0
team_id          0
start_season     0
end_season       0
start_year       0
end_year        26
source           0

Sample records:


,owner_name,team_id,start_season,end_season,start_year,end_year,source
0,Leslie Alexander,HOU,1993-1994,2016-2017,1993,2016.0,RealGM ownership CSV
1,Paul Allen,POR,1988-1989,2017-2018,1988,2017.0,RealGM ownership CSV
2,Steve Ballmer,LAC,2014-2015,-,2014,NaN,RealGM ownership CSV
3,Clay Bennett,OKC,2006-2007,-,2006,NaN,RealGM ownership CSV
4,Tom Benson,NOP,2011-2012,2017-2018,2011,2017.0,RealGM ownership CSV
5,Sara Blakely,ATL,2015-2016,-,2015,NaN,RealGM ownership CSV
6,Buss Family Trust,LAL,2013-2014,2024-2025,2013,2024.0,RealGM ownership CSV
7,Dr. Jerry H. Buss,LAL,1979-1980,2012-2013,1979,2012.0,RealGM ownership CSV


In [3]:
print("S15 exact duplicate rows:", s15.duplicated().sum())
print("S16 exact duplicate rows:", s16.duplicated().sum())

print("\nS15 franchise identifiers:")
print(sorted(s15["franchise_id"].dropna().unique()))

print("\nS16 team identifiers:")
print(sorted(s16["team_id"].dropna().astype(str).unique()))

S15 exact duplicate rows: 0
S16 exact duplicate rows: 0

S15 franchise identifiers:
['ATL', 'BOS', 'BRK', 'CHI', 'CHO', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHO', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']

S16 team identifiers:
['ATL', 'BOS', 'BRK', 'CHI', 'CHO', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHO', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']


In [4]:
franchises = pd.read_csv(
    CANONICAL_DIR / "franchises.csv"
)

print(franchises.columns.tolist())
display(franchises.head())

['franchise_id', 'current_team_name', 'current_abbreviation']


,franchise_id,current_team_name,current_abbreviation
0,1610612737,Atlanta Hawks,ATL
1,1610612738,Boston Celtics,BOS
2,1610612739,Cleveland Cavaliers,CLE
3,1610612740,New Orleans Pelicans,NOP
4,1610612741,Chicago Bulls,CHI


In [5]:
# S15: convert exact dates to datetime.
gm_dates = s15.copy()

gm_dates["start_date"] = pd.to_datetime(
    gm_dates["start_date"],
    errors="coerce"
)

gm_dates["end_date"] = pd.to_datetime(
    gm_dates["end_date"],
    errors="coerce"
)

print("S15 invalid start dates:",
      gm_dates["start_date"].isna().sum())

print("S15 missing or invalid end dates:",
      gm_dates["end_date"].isna().sum())

print("S15 end dates before start dates:",
      (
          gm_dates["end_date"].notna()
          & (gm_dates["end_date"] < gm_dates["start_date"])
      ).sum())


# S16: validate the season-year representation.
owner_dates = s16.copy()

owner_dates["start_year_from_season"] = pd.to_numeric(
    owner_dates["start_season"].str[:4],
    errors="coerce"
)

owner_dates["end_year_from_season"] = pd.to_numeric(
    owner_dates["end_season"].str[:4],
    errors="coerce"
)

print("\nS16 invalid start seasons:",
      owner_dates["start_year_from_season"].isna().sum())

print("S16 ongoing ownership records:",
      owner_dates["end_year"].isna().sum())

print("S16 start-year mismatches:",
      (
          owner_dates["start_year_from_season"]
          != owner_dates["start_year"]
      ).sum())

print("S16 end-year mismatches:",
      (
          owner_dates["end_year"].notna()
          & (
              owner_dates["end_year_from_season"]
              != owner_dates["end_year"]
          )
      ).sum())

print("S16 end years before start years:",
      (
          owner_dates["end_year"].notna()
          & (owner_dates["end_year"] < owner_dates["start_year"])
      ).sum())

S15 invalid start dates: 0
S15 missing or invalid end dates: 30
S15 end dates before start dates: 0

S16 invalid start seasons: 0
S16 ongoing ownership records: 26
S16 start-year mismatches: 0
S16 end-year mismatches: 0
S16 end years before start years: 0


In [6]:
print("S15 records per franchise:")
display(
    s15.groupby("franchise_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

print("\nS16 records per franchise:")
display(
    s16.groupby("team_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

print("\nS16 example: Atlanta ownership")
display(
    s16[s16["team_id"] == "ATL"]
    .sort_values("start_year")
)

print("\nS15 example: Atlanta executives")
display(
    s15[s15["franchise_id"] == "ATL"]
    .sort_values("start_date")
)

S15 records per franchise:


franchise_id
PHI    12
MIN    10
NYK     9
POR     9
ATL     8
PHO     8
BRK     8
LAC     8
MEM     7
ORL     7
dtype: int64


S16 records per franchise:


team_id
ATL    7
BRK    5
BOS    5
UTA    4
DET    3
MIN    3
PHO    3
GSW    3
IND    3
CLE    3
dtype: int64


S16 example: Atlanta ownership


,owner_name,team_id,start_season,end_season,start_year,end_year,source
66,Ted Turner,ATL,1976-1977,2004-2005,1976,2004.0,RealGM ownership CSV
38,Bruce Levenson,ATL,2005-2006,2014-2015,2005,2014.0,RealGM ownership CSV
5,Sara Blakely,ATL,2015-2016,-,2015,NaN,RealGM ownership CSV
15,Clayton Dubilier,ATL,2015-2016,-,2015,NaN,RealGM ownership CSV
27,Grant Hill,ATL,2015-2016,-,2015,NaN,RealGM ownership CSV
31,Jesse Itzler,ATL,2015-2016,-,2015,NaN,RealGM ownership CSV
53,Rick Schnall,ATL,2015-2016,2022-2023,2015,2022.0,RealGM ownership CSV



S15 example: Atlanta executives


,executive_name,start_date,end_date,franchise_id,franchise_name
0,Pete Babcock,1990-02-14,2003-04-02,ATL,Atlanta Hawks
1,Billy Knight,2003-04-02,2008-05-28,ATL,Atlanta Hawks
2,Rick Sund,2008-05-28,2012-06-25,ATL,Atlanta Hawks
3,Danny Ferry,2012-06-25,2014-09-12,ATL,Atlanta Hawks
4,Mike Budenholzer,2014-09-12,2017-05-25,ATL,Atlanta Hawks
5,Travis Schlenk,2017-05-25,2022-12-21,ATL,Atlanta Hawks
6,Landry Fields,2022-12-21,2025-04-21,ATL,Atlanta Hawks
7,Onsi Saleh,2025-04-21,NaN,ATL,Atlanta Hawks


In [7]:
franchises = pd.read_csv(
    CANONICAL_DIR / "franchises.csv"
)

print(franchises.columns.tolist())
display(franchises.head(3))

['franchise_id', 'current_team_name', 'current_abbreviation']


,franchise_id,current_team_name,current_abbreviation
0,1610612737,Atlanta Hawks,ATL
1,1610612738,Boston Celtics,BOS
2,1610612739,Cleveland Cavaliers,CLE


In [8]:
# C06 — Leadership tenures
# Grain: one entity × franchise × role × tenure interval

franchise_map = franchises[
    ["franchise_id", "current_abbreviation"]
].copy()

# S15: executive tenures, with exact dates
executives = (
    s15.rename(columns={
        "executive_name": "leader_name",
        "franchise_id": "team_abbreviation"
    })
    .assign(
        role="executive",
        source_table="S15",
        start_date=lambda df: pd.to_datetime(
            df["start_date"], errors="coerce"
        ),
        end_date=lambda df: pd.to_datetime(
            df["end_date"], errors="coerce"
        ),
        start_season=pd.NA,
        end_season=pd.NA
    )
)

executives = executives.merge(
    franchise_map,
    left_on="team_abbreviation",
    right_on="current_abbreviation",
    how="left",
    validate="many_to_one"
)

# S16: ownership tenures, with season-level precision
owners = (
    s16.rename(columns={
        "owner_name": "leader_name",
        "team_id": "team_abbreviation"
    })
    .assign(
        role="owner",
        source_table="S16",
        start_date=pd.NaT,
        end_date=pd.NaT
    )
)

owners = owners.merge(
    franchise_map,
    left_on="team_abbreviation",
    right_on="current_abbreviation",
    how="left",
    validate="many_to_one"
)

# Retain the original source precision and provenance
columns = [
    "franchise_id",
    "leader_name",
    "role",
    "start_date",
    "end_date",
    "start_season",
    "end_season",
    "source_table"
]

leadership_tenures = pd.concat(
    [
        executives[columns],
        owners[columns]
    ],
    ignore_index=True
)

print("C06 shape:", leadership_tenures.shape)
display(leadership_tenures.head(5))
display(leadership_tenures.tail(5))

C06 shape: (251, 8)


,franchise_id,leader_name,role,start_date,end_date,start_season,end_season,source_table
0,1.610613e+09,Pete Babcock,executive,1990-02-14,2003-04-02,<NA>,<NA>,S15
1,1.610613e+09,Billy Knight,executive,2003-04-02,2008-05-28,<NA>,<NA>,S15
2,1.610613e+09,Rick Sund,executive,2008-05-28,2012-06-25,<NA>,<NA>,S15
3,1.610613e+09,Danny Ferry,executive,2012-06-25,2014-09-12,<NA>,<NA>,S15
4,1.610613e+09,Mike Budenholzer,executive,2014-09-12,2017-05-25,<NA>,<NA>,S15


,franchise_id,leader_name,role,start_date,end_date,start_season,end_season,source_table
246,NaN,Joseph Tsai,owner,NaT,NaT,2017-2018,-,S16
247,1.610613e+09,Ted Turner,owner,NaT,NaT,1976-1977,2004-2005,S16
248,1.610613e+09,Mark Cuban,owner,NaT,NaT,2000-2001,2023-2024,S16
249,1.610613e+09,Adelson-Dumont family,owner,NaT,NaT,2023-2024,-,S16
250,1.610613e+09,Micky Arison,owner,NaT,NaT,1995-1996,-,S16


In [9]:
# Row preservation
print("Source rows:", len(s15) + len(s16))
print("Canonical rows:", len(leadership_tenures))

# Franchise identity
print(
    "Missing franchise IDs:",
    leadership_tenures["franchise_id"].isna().sum()
)

# Leadership categories
print("\nRoles:")
print(leadership_tenures["role"].value_counts())

# Missing end values (expected for open-ended tenures)
print("\nExecutive missing end dates:",
    leadership_tenures.loc[
        leadership_tenures["role"] == "executive",
        "end_date"
    ].isna().sum()
)

print("Owner missing end seasons:",
    leadership_tenures.loc[
        leadership_tenures["role"] == "owner",
        "end_season"
    ].isin(["-", ""]).sum()
)

# Duplicate canonical records
print(
    "\nExact duplicate records:",
    leadership_tenures.duplicated().sum()
)

# Date consistency for executives
executive_records = leadership_tenures[
    leadership_tenures["role"] == "executive"
]

print(
    "Executive end dates before start:",
    (
        executive_records["end_date"].notna()
        & (
            executive_records["end_date"]
            < executive_records["start_date"]
        )
    ).sum()
)

# Check start dates and ownership season values
print(
    "Executive missing start dates:",
    executive_records["start_date"].isna().sum()
)

owner_records = leadership_tenures[
    leadership_tenures["role"] == "owner"
]

print(
    "Owner missing start seasons:",
    owner_records["start_season"].isna().sum()
)

assert len(leadership_tenures) == len(s15) + len(s16)
assert leadership_tenures["franchise_id"].notna().all()
assert leadership_tenures.duplicated().sum() == 0

Source rows: 251
Canonical rows: 251
Missing franchise IDs: 32

Roles:
role
executive    181
owner         70
Name: count, dtype: int64

Executive missing end dates: 30
Owner missing end seasons: 26

Exact duplicate records: 0
Executive end dates before start: 0
Executive missing start dates: 0
Owner missing start seasons: 0


AssertionError: 

In [10]:
unmatched = leadership_tenures[
    leadership_tenures["franchise_id"].isna()
]

print("Unmatched records:", len(unmatched))

print("\nUnmatched abbreviations:")
print(
    unmatched.groupby(
        ["team_abbreviation", "role"]
    ).size()
)

Unmatched records: 32

Unmatched abbreviations:


KeyError: 'team_abbreviation'

In [11]:
print(leadership_tenures.columns.tolist())

['franchise_id', 'leader_name', 'role', 'start_date', 'end_date', 'start_season', 'end_season', 'source_table']


In [12]:
# Compare source abbreviations with the canonical franchise lookup.

source_abbreviations = pd.concat([
    s15["franchise_id"],
    s16["team_id"]
]).unique()

canonical_abbreviations = set(
    franchises["current_abbreviation"]
)

unmatched_abbreviations = sorted(
    set(source_abbreviations) - canonical_abbreviations
)

print("Unmatched abbreviations:", unmatched_abbreviations)

# Count the records affected.
for abbr in unmatched_abbreviations:
    gm_count = (s15["franchise_id"] == abbr).sum()
    owner_count = (s16["team_id"] == abbr).sum()

    print(
        f"{abbr}: {gm_count} executive records, "
        f"{owner_count} ownership records"
    )

Unmatched abbreviations: ['BRK', 'CHO', 'PHO']
BRK: 8 executive records, 5 ownership records
CHO: 6 executive records, 2 ownership records
PHO: 8 executive records, 3 ownership records


In [13]:
# C06: Correct abbreviation differences between source and canonical data.

abbr_corrections = {
    "BRK": "BKN",
    "CHO": "CHA",
    "PHO": "PHX"
}

# Verify that these target abbreviations exist in C02.
assert set(abbr_corrections.values()).issubset(
    set(franchises["current_abbreviation"])
), "Check the canonical abbreviations before proceeding."

franchise_map = franchises[
    ["franchise_id", "current_abbreviation"]
].copy()

# Executive records
executives = s15.copy()

executives["team_abbreviation"] = (
    executives["franchise_id"].replace(abbr_corrections)
)

executives = executives.rename(
    columns={"executive_name": "leader_name"}
)

executives["role"] = "executive"
executives["source_table"] = "S15"
executives["start_date"] = pd.to_datetime(executives["start_date"])
executives["end_date"] = pd.to_datetime(executives["end_date"])
executives["start_season"] = pd.NA
executives["end_season"] = pd.NA

# Ownership records
owners = s16.copy()

owners["team_abbreviation"] = (
    owners["team_id"].replace(abbr_corrections)
)

owners = owners.rename(
    columns={"owner_name": "leader_name"}
)

owners["role"] = "owner"
owners["source_table"] = "S16"
owners["start_date"] = pd.NaT
owners["end_date"] = pd.NaT

# Combine source records
leadership_raw = pd.concat(
    [
        executives,
        owners
    ],
    ignore_index=True
)

# Map to numeric franchise IDs
leadership_mapped = leadership_raw.merge(
    franchise_map,
    left_on="team_abbreviation",
    right_on="current_abbreviation",
    how="left",
    validate="many_to_one"
)

# Select final canonical columns
columns = [
    "franchise_id",
    "leader_name",
    "role",
    "start_date",
    "end_date",
    "start_season",
    "end_season",
    "source_table"
]

leadership_tenures = leadership_mapped[columns].copy()

print("C06 shape:", leadership_tenures.shape)
print("Missing franchise IDs:",
      leadership_tenures["franchise_id"].isna().sum())

KeyError: "['franchise_id'] not in index"

In [14]:
# Rename the numeric C02 franchise ID to our canonical name.
leadership_mapped = leadership_mapped.rename(
    columns={"franchise_id_y": "franchise_id"}
)

# Select the canonical columns.
columns = [
    "franchise_id",
    "leader_name",
    "role",
    "start_date",
    "end_date",
    "start_season",
    "end_season",
    "source_table"
]

leadership_tenures = leadership_mapped[columns].copy()

print("Shape:", leadership_tenures.shape)
print("Missing franchise IDs:", leadership_tenures["franchise_id"].isna().sum())
print("Roles:")
print(leadership_tenures["role"].value_counts())

Shape: (251, 8)
Missing franchise IDs: 0
Roles:
role
executive    181
owner         70
Name: count, dtype: int64


In [15]:
assert len(leadership_tenures) == 251
assert leadership_tenures["franchise_id"].notna().all()
assert leadership_tenures.duplicated().sum() == 0
assert (
    leadership_tenures["role"].value_counts().to_dict()
    == {"executive": 181, "owner": 70}
)

output_path = CANONICAL_DIR / "leadership_tenures.csv"
leadership_tenures.to_csv(output_path, index=False)

print("C06 exported successfully:", output_path)
print("Final shape:", leadership_tenures.shape)

C06 exported successfully: /Users/jonathantran/Desktop/nba_economics/data/canonical/leadership_tenures.csv
Final shape: (251, 8)


In [16]:
executives = leadership_tenures.query("role == 'executive'")
owners = leadership_tenures.query("role == 'owner'")